# How one country gets its risk rating

One country, start to finish, in the order `backend/util/pipeline.py` does it:
**fetch** the raw data, **process** it into the evidence payload, hand the model
exactly what it gets to read, and unpack what it says back.

Every cell calls the production function. Nothing here is a re-implementation,
and nothing here is a simplification of the real path — in particular this
notebook runs **masked**, which is what `_process_country` does by default: the
model is never told which country it is scoring.

**What it costs to run:** one cheap digest call per article (~20), up to three
body-rewrite calls, and one scoring call. Roughly a dime.

**What it writes:** the `llm_artifact` digest and rewrite caches, if
`RISK_DB_TARGET` is set. Nothing else — `upsert_snapshot` is never called, so no
row lands in `risk_snapshot` and the dashboard does not move.

**To run:** set `ISO2` two cells down, then Run All. Kernel, once:

```
.venv\Scripts\python.exe -m pip install ipykernel
```

In [ ]:
import json
import logging
import os
import pathlib
import sys

import pandas as pd
from dotenv import load_dotenv

# Repo root = the folder holding backend/main.py, so this works from any cwd.
PROJECT_ROOT = next(
    p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    if (p / "backend" / "main.py").exists()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / "backend" / ".env")
load_dotenv()

# force=True is not optional: Jupyter installs its own root handler, so a plain
# basicConfig() is silently a no-op and no pipeline logs ever appear.
logging.basicConfig(level=logging.INFO, format="%(levelname)-7s %(name)s: %(message)s",
                    stream=sys.stdout, force=True)
# httpx logs a line per request and per redirect - fifty of them for one news
# fetch, none about risk.
logging.getLogger("httpx").setLevel(logging.WARNING)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

from backend.util import constants, lint, pipeline, policy, provenance
from backend.llm import client as ai_client
from backend.llm import constants as ai_constants
from backend.llm import digest_engine, gazetteer, langchain_llm, probe, rewrite
from backend.llm import payload as llm_payload
from backend.data_fetching import (
    bis_bulk_fetch, curated_loader, imf_macro_fetch, wb_series_fetch,
)
from backend.data_upsert import data_push
from backend.news_fetching import article_enrichment, article_ranking, core as news_core

# Imported, not copied: a local copy of these drifts from the pipeline silently.
SINCE_YEAR    = pipeline._PAYLOAD_SINCE_YEAR
LOOKBACK      = pipeline._PAYLOAD_LOOKBACK_YEARS
DELTAS        = pipeline._PAYLOAD_DELTA_HORIZONS
MAX_ARTICLES  = pipeline._MAX_ARTICLES_PER_COUNTRY

# The database is chosen explicitly. There is no default: `RISK_DB_TARGET` must
# say 'prod' or 'dev', and the URL var it names has to be set.
try:
    DB_TARGET = data_push.resolve_target()
    DB_VAR = data_push._TARGETS[DB_TARGET]
    db_state = f"{DB_TARGET} -> {DB_VAR} {'set' if os.getenv(DB_VAR) else 'MISSING'}"
except RuntimeError as exc:
    DB_TARGET, db_state = None, f"unset ({exc})"

print("environment")
for key in ("OPENAI_API_KEY", "CRAWLBASE_TOKEN"):
    print(f"  {key:<18} {'set' if os.getenv(key) else 'MISSING'}")
print(f"  {'RISK_DB_TARGET':<18} {db_state}")

print("\nversions the run is stamped with")
print(f"  scoring   {ai_client.scoring_model()}  temp 0.0  seed {ai_client.SEED}")
print(f"  digest    {ai_client.digest_model()}")
print(f"  prompt    {ai_constants.PROMPT_VERSION}   policy {policy.POLICY_VERSION}")
print(f"  masking   map {gazetteer.MASK_MAP_VERSION}  gazetteer {gazetteer.GAZETTEER_VERSION[:12]}"
      f"  sweep {rewrite.SWEEP_VERSION[:12]}")
print(f"  payload   variant {provenance.payload_variant()}   prompt variant "
      f"{provenance.prompt_variant() or '(none)'}")
print(f"\n{len(constants.INDICATOR_REGISTRY)} indicators in the registry, across 4 ledgers")

## Pick a country

The roster is `constants.COUNTRY_ROSTER`. Everything below keys off this one
choice.

In [ ]:
ISO2 = "PT"   # <-- change country here

ENTRY = next(c for c in constants.COUNTRY_ROSTER if c["iso2"] == ISO2)
NAME, ISO3 = ENTRY["name"], ENTRY["iso3"]

print(f"{NAME}  ({ISO2}/{ISO3})  tier={ENTRY['tier']}")
print(f"{len(constants.COUNTRY_ROSTER)} countries in the roster")
pd.DataFrame(constants.COUNTRY_ROSTER)[["name", "iso2", "iso3", "tier"]].head(8)

---
# A · Fetch

Raw bytes from upstream. Nothing is derived here — no scoring, no shaping, no
model. Four sources feed a country: the numeric series, the news, the article
bodies behind that news, and a handful of static files.

### A1 · The numeric series

Every number the model ever sees lives in one Postgres table,
`indicator_series` — World Bank annuals, IMF monthlies, BIS policy rates and FX,
and the curated CSV drop, all in the same row shape. If this country already has
rows, they are used as-is; if not, the four fetchers run.

`refresh_ledger_sources()` and `refresh_imf_indicators()` are what does this in
production, roster-wide, once per run.

In [ ]:
FETCH_IF_EMPTY = True     # False = score on whatever is already stored
FETCH_BIS      = True     # ~14 MB of bulk files; the only source of FX volatility


def safe(read, what):
    """The pipeline's own per-store resilience: a failed read is absent, not fatal."""
    try:
        return read()
    except Exception as exc:
        print(f"  {what:<20} unavailable ({type(exc).__name__}) - degrading to absent")
        return None


series = safe(lambda: data_push.read_indicator_series(ISO2), "indicator_series") or {}

if not series and FETCH_IF_EMPTY:
    print(f"nothing stored for {ISO2} - fetching from upstream")
    rows = []
    rows += safe(lambda: wb_series_fetch.fetch_country_series(ISO2, ISO3), "world bank") or []
    rows += safe(lambda: imf_macro_fetch.fetch_series_rows(ISO2, ISO3), "imf") or []
    if FETCH_BIS:
        for code_ in ("BIS.POLICY.RATE", "BIS.FX.USD"):
            rows += [r for r in safe(lambda: bis_bulk_fetch.fetch_dataset_rows(code_), f"bis {code_}") or []
                     if r["country_iso2"] == ISO2]
    rows += [r for r in safe(curated_loader.load_curated_series, "curated.csv") or []
             if r["country_iso2"] == ISO2]
    # Deliberately NOT upserted. This notebook writes no series rows; the daily
    # run owns that table.
    for r in sorted(rows, key=lambda r: (r["indicator_code"], r["period"])):
        series.setdefault(r["indicator_code"], []).append(r)

print(f"\n{len(series)} indicator(s), {sum(len(v) for v in series.values())} raw observation(s)")
pd.DataFrame([
    {"code": code_, "rows": len(rows_), "first": rows_[0]["period"], "last": rows_[-1]["period"],
     "source": rows_[-1].get("source"),
     "ledger": (constants.INDICATOR_REGISTRY.get(code_) or {}).get("ledger")}
    for code_, rows_ in sorted(series.items())
]).set_index("code")

### A2 · The news

Six Google News queries, one per theme the prompt scores plus a broad catch-all.
Results are de-duplicated on publisher URL, scored by
`article_ranking.score_relevance`, and selected with a per-theme floor so no one
query takes the whole budget.

In [ ]:
print(f"{len(news_core.THEME_QUERIES)} queries, one per theme:\n")
for theme, template in news_core.THEME_QUERIES.items():
    print(f"  {theme:<12} {template.format(c=NAME)[:110]}")

items = article_enrichment.fetch_relevant_news(NAME, max_articles=MAX_ARTICLES)
print(f"\n{len(items)} article(s) survived de-duplication and selection")

pd.DataFrame([
    {"theme": it.get("_theme"), "relevance": round(it.get("relevance_score", 0), 2),
     "source": it.get("source"), "published": (it.get("published") or "")[:10],
     "title": it.get("title")}
    for it in items
]).sort_values("relevance", ascending=False)

### A3 · Resolve the wrappers, fetch the bodies

What A2 returned are `news.google.com` redirect wrappers with a feed blurb.
`resolve_and_enrich` unwraps each to its publisher URL, drops denylisted
sources, and scrapes the body. Bodies are capped at
`core.MAX_BODY_CHARS` so a historical article and a live one are the same size
of evidence.

Ids `a1..aN` are assigned right after, and they are what the model cites back.

In [ ]:
n_before = len(items)
sample = items[0]
print(f"BEFORE   link  {sample.get('link', '')[:100]}")
print(f"         body  {len(sample.get('text') or ''):,} chars")

items = article_enrichment.resolve_and_enrich(items, ISO2)
for i, it in enumerate(items, start=1):
    it["id"] = f"a{i}"          # the stable ids the model cites back

print(f"\nAFTER    link  {items[0].get('link', '')[:100]}")
print(f"         body  {len(items[0].get('text') or ''):,} chars")
print(f"\n{n_before} in -> {len(items)} out (denylisted sources dropped)")
print(f"body cap: {news_core.MAX_BODY_CHARS:,} chars")

pd.DataFrame([
    {"id": it["id"], "chars": len(it.get("text") or ""), "has_image": bool(it.get("image")),
     "source": it.get("source"), "url": (it.get("link") or "")[:80]}
    for it in items
]).set_index("id")

### A4 · The static evidence

Three things that are not fetched from anywhere: the structural facts file, the
FX regime table, and the election calendar. The structural block exists
*because* of masking — it states the priors the country's name would have
carried, so the model can reason from them instead of from a guess.

In [ ]:
structural = safe(curated_loader.load_structural_facts, "structural_facts.yaml") or {}

print(f"structural facts: {len(structural)} of {len(constants.COUNTRY_ROSTER)} countries have a block")
print(f"  {ISO2}: {json.dumps(structural.get(ISO2), indent=2) if structural.get(ISO2) else 'none - treated as unknown, never guessed'}")
print(f"\nfx regime:  {constants.FX_REGIMES.get(ISO2)}")
print(f"elections:  {constants.ELECTIONS.get(ISO2, [])}")

---
# B · Process

Seven transforms turn what A fetched into what the model reads. Each gets its
own cell, in pipeline order.

### B1 · The panel payload — and why the scorer never sees it

`prepare_llm_payload_pretty` builds the wide annual panel. **This is not what
the model scores on.** It exists because `upsert_snapshot` reads its
`indicators` and `_meta.units` to write the tables the front-end's indicator
pane queries.

It does own one thing the whole run depends on: `_meta.generated_at`, which
`payload_as_of` parses back into `AS_OF` — the date the snapshot is keyed on,
the date the prompt is anchored to, and the date the sanctions rules are
evaluated against.

In [ ]:
panel = llm_payload.prepare_llm_payload_pretty(
    country_iso=ISO2, indicators=constants.ALL_INDICATORS,
    since=SINCE_YEAR, lookback=LOOKBACK, deltas=DELTAS,
)
AS_OF = data_push.payload_as_of(panel)

print(f"AS_OF = {AS_OF}   (from _meta.generated_at = {panel['_meta']['generated_at']})")
print(f"latest_year {panel['latest_year']}, {len(panel['indicators'])} indicators, "
      f"{LOOKBACK}y series, deltas {DELTAS}\n")

pd.DataFrame([
    {"indicator": k, "latest": v["latest"], **{d: v.get(d) for d in (f"\u0394{h}y" for h in DELTAS)},
     "years": len(v["series"])}
    for k, v in panel["indicators"].items()
]).set_index("indicator").head(12)

### B2 · Raw rows become stamped observations

The core of the shaping step, on one indicator. Raw rows carry a `period` and a
`freq`; `_resolve` picks the freshest observation per period across sources and
(for a backfill) drops anything published after the vintage bound; `_stamp` adds
`period`, `freq`, `as_of`, `staleness_days` and `source`.

`staleness_days` counts from the **end of the period a value describes** to
`AS_OF` — how old the reading is. `as_of` is a separate fact: when it became
known to us.

In [ ]:
DEMO = next((c for c in series if c in constants.INDICATOR_REGISTRY), None)

raw = series[DEMO]
obs = llm_payload._series_observations(raw)
resolved = llm_payload._resolve(obs, None)          # None = no vintage bound (the live run)
stamped = llm_payload._stamp(resolved, DEMO, AS_OF)

print(f"{DEMO}  -  {constants.INDICATOR_REGISTRY[DEMO]['label']}")
print(f"  ledger: {constants.INDICATOR_REGISTRY[DEMO].get('ledger')}\n")
print(f"raw rows       {len(raw):>4}   e.g. {json.dumps(raw[-1], default=str)[:150]}")
print(f"observations   {len(obs):>4}   e.g. {obs[-1]}")
print(f"resolved       {len(resolved):>4}   (freshest per period wins)\n")
print("stamped, which is what the payload carries:")
print(json.dumps(stamped, indent=2, default=str))

### B3 · The evidence payload — the thing the model actually scores

`build_evidence_payload` resolves every registry indicator the way B2 just did,
buckets each into its ledger, and adds a `computed` block of derived metrics
from `backend/util/metrics.py`.

Every store is passed **in** rather than read inside, which is what makes it
re-runnable over history: the caller decides which snapshot of the world it
sees. An indicator with no observation is omitted entirely — absent means
absent, never zero.

In [ ]:
evidence = llm_payload.build_evidence_payload(
    ISO2,
    as_of=AS_OF,
    series=series,
    fx_regimes=constants.FX_REGIMES,
    elections=constants.ELECTIONS,
    structural=structural,
    vintage_as_of=None,      # a historical backfill passes AS_OF here; the live run does not
)

LEDGERS = ["friction_inputs", "uncertainty_inputs", "information_inputs", "edge_inputs"]
for section in LEDGERS:
    keys = list(evidence.get(section, {}))
    print(f"{section:<22} {len(keys):>2}  {', '.join(keys[:4])}{' ...' if len(keys) > 4 else ''}")

print(f"\ncomputed ({len(evidence['computed'])} metrics from backend/util/metrics.py):")
print(json.dumps(evidence["computed"], indent=2, default=str))
print(f"\nstructural block: {'present' if evidence.get('structural') else 'absent'}")

In [ ]:
# One ledger unrolled, so the per-value stamps are visible.
pd.DataFrame([
    {"indicator": k, "value": v.get("value"), "period": v.get("period"),
     "freq": v.get("freq"), "stale_days": v.get("staleness_days"), "source": v.get("source")}
    for k, v in evidence["friction_inputs"].items() if isinstance(v, dict)
]).set_index("indicator")

### B4 · Masking, pass one

Production scores every country **without naming it**. The gazetteer replaces
country names, cities, people, parties, currencies and institutions with the
roles they play — "the country", "the capital", "the central bank". Every number
is untouched.

It happens here rather than at the model call because a digest written from
named text carries the name into the prompt however clean the article beside it
is. From this line to the score, everything reads `scored`; everything the
database and the front end read stays on `items`, unmasked.

In [ ]:
scored = rewrite.mask_items(items, ISO2)
DISPLAY = langchain_llm.MASKED_COUNTRY_LABEL      # "the country"

demo_i = next((i for i, it in enumerate(items)
               if gazetteer.mentions(it.get("title") or "", ISO2)), 0)
print(f"country label in the prompt: {DISPLAY!r}\n")
print(f"BEFORE  {items[demo_i]['title']}")
print(f"AFTER   {scored[demo_i]['title']}\n")
print(f"BEFORE  {(items[demo_i].get('text') or '')[:300]}")
print(f"\nAFTER   {(scored[demo_i].get('text') or '')[:300]}")

### B5 · Stage one — every article becomes a digest

The cheap model reads each article's full body and returns a small structured
object. That is what keeps the scoring prompt bounded: ~20 bodies would be tens
of thousands of tokens, and the scorer reads digests instead.

`masked=True` is about what the digest model *writes*, not what it reads — the
text is already masked. Without it, `actors: who did what to whom` reads as an
instruction to name people, and people are exactly what a gazetteer cannot know.

Cached on content hash in `llm_artifact`, so a re-run of this notebook costs
nothing here.

In [ ]:
print("DIGEST_PROMPT (backend/llm/constants.py), with the mask rule appended:")
print("=" * 78)
print(ai_constants.DIGEST_PROMPT.format(
    country=DISPLAY, article_text="<the article body goes here>",
    mask_rule=ai_constants.DIGEST_MASK_RULE))
print("=" * 78)

In [ ]:
scored = digest_engine.digest_articles(
    scored, country_display=DISPLAY, iso2=ISO2, as_of=AS_OF, masked=True,
)
digested = [it for it in scored if isinstance(it.get("digest"), dict)]
print(f"\n{len(digested)}/{len(scored)} digested. A failure leaves digest=None and that "
      f"article degrades to title+summary in the prompt.")

focus = max(digested, key=lambda it: it.get("stage1_severity") or 0.0)
print(f"\nONE DIGEST IN FULL - {focus['id']}, the highest-severity article "
      f"({len(digest_engine.article_input_text(focus)):,} chars in, "
      f"{len(json.dumps(focus['digest'])):,} out)\n")
print(json.dumps(focus["digest"], indent=2, ensure_ascii=False))

### B6 · Which three the scorer reads end to end

Every digest reaches the model. This step is about **depth, not inclusion**:
`select_fulltext_ids` picks the three highest-severity articles whose full text
gets pasted into the prompt as well.

In [ ]:
fulltext_ids = digest_engine.select_fulltext_ids(scored)
print(f"full text goes to the scorer for: {fulltext_ids}\n")

pd.DataFrame([
    {"id": it["id"], "severity": it.get("stage1_severity"),
     "full_text": it["id"] in fulltext_ids, "source": it.get("source"),
     "what_happened": (it.get("digest") or {}).get("what_happened")}
    for it in scored
]).sort_values("severity", ascending=False).set_index("id")

### B7 · Masking, pass two — the model masks what the list could not

The gazetteer is a list somebody wrote. It does not know this week's finance
minister, this year's ruling party, or the bank that just failed — and those are
named in a full body far more often than the country is. So the three bodies the
scorer reads end to end get a second pass, by a model.

**It fails closed.** A rewrite that errors or comes back empty leaves the article
with no body, and it reaches the scorer as its masked title. Being short one body
costs a week some evidence; one leaked name costs the whole comparison.

In [ ]:
was = {aid: len(digest_engine.article_input_text(it))
       for aid in fulltext_ids
       for it in scored if it.get("id") == aid}

pipeline._rewrite_fulltext(scored, fulltext_ids, ISO2)   # the production function

for it in scored:
    if it.get("id") in fulltext_ids:
        now = len(digest_engine.article_input_text(it))
        state = "title-only (failed closed)" if now == 0 else "rewritten"
        print(f"  {it['id']}  {was[it['id']]:>6,} -> {now:>6,} chars   {state}")

---
# C · Everything the model sees

One `SystemMessage`. No user turn, no tool definitions, no conversation. The
whole prompt is assembled below, byte for byte, with the same four builders
`country_llm_score` uses at `backend/llm/langchain_llm.py:400-429`.

### C1 · The second masking, and the gate

`country_llm_score(mask_iso2=...)` masks again on its way out — the evidence
payload names the country in its `_meta` and its series labels, and the digests
were generated after pass one. Then `assert_clean` scans the four strings that
are about to be sent and **raises** if any roster country's name survived.

Not the whole prompt: the template's own worked examples name Australia and
China, and those are instructions rather than evidence about anyone. These four
strings are every byte the prompt carries that came from this country's data.

In [ ]:
payload_sent  = rewrite.mask_payload(evidence, ISO2)
articles_sent = rewrite.mask_items(scored, ISO2)

evidence_json  = json.dumps(payload_sent, ensure_ascii=False)
articles_json  = langchain_llm._digests_to_json(articles_sent)
fulltext_block = langchain_llm._fulltext_block(articles_sent, fulltext_ids)

# Raises MaskLeak if a roster country name survived. This is the notebook's test.
rewrite.assert_clean([evidence_json, articles_json, fulltext_block, DISPLAY])
print("mask integrity: clean - no roster country name in anything about to be sent")

# The sanctions lookup keeps the REAL code; it is read off the payload before
# masking, which is why _extract_iso2 runs first inside the call.
print(f"sanctions lookup still uses {ISO2}; the prompt never learns it")

### C2 · The four slots

`AI_PROMPT_V3` has five placeholders: the country label, the date, and these
three JSON/text blocks.

In [ ]:
print(f"country       {DISPLAY!r}")
print(f"as_of_date    {AS_OF.isoformat()}   (treated as 'today' by the prompt)\n")
for label, blob in (("EVIDENCE_JSON", evidence_json),
                    ("ARTICLES_JSON", articles_json),
                    ("FULL_TEXT", fulltext_block)):
    print(f"{label:<16}{len(blob):>9,} chars  ~{len(blob)//4:>6,} tokens")
print(f"\nfor comparison, every article body in full: "
      f"{sum(len(digest_engine.article_input_text(it)) for it in articles_sent):,} chars "
      f"- what the digests replaced")

print("\n" + "-" * 78 + "\nARTICLES_JSON, first entry:\n")
print(json.dumps(json.loads(articles_json)[0], indent=2, ensure_ascii=False))

### C3 · The prompt, verbatim

This is the entire string the model receives. Nothing else is sent.

In [ ]:
rules, PROMPT_VERSION = langchain_llm._prompt_rules_and_version(payload_sent)

PROMPT = ai_constants.AI_PROMPT_V3.format(
    country=DISPLAY,
    as_of_date=AS_OF.isoformat(),
    evidence_json=evidence_json,
    articles_json=articles_json,
    full_text_block=fulltext_block if fulltext_block != "(none)"
    else "(no full-text articles supplied)",
) + rules

print(f"prompt_version stamped: {PROMPT_VERSION}")
print(f"appended rule blocks:   {len(rules):,} chars"
      f"{' (none - base template)' if not rules else ''}")
print(f"total prompt:           {len(PROMPT):,} chars  ~{len(PROMPT)//4:,} tokens")
print("\n" + "=" * 78)
print(PROMPT)
print("=" * 78)

### C4 · What it is asked to return, and how

The schema is bound with `strict=True`, so the model cannot return a field the
schema does not name. `strict` enforces *shape*, not `minimum`/`maximum` — which
is why `_from_100` clamps on the way back.

In [ ]:
print(f"model         {ai_client.scoring_model()}")
print(f"temperature   0.0     seed {ai_client.SEED}     max_retries 0")
print(f"messages      [SystemMessage(content=<the prompt above>)]   - no user turn")
print(f"tools         none")
print(f"schema        RISK_SCHEMA_V3, strict=True\n")

schema = langchain_llm._SCHEMA_BY_PROMPT_VARIANT.get(
    provenance.prompt_variant(), ai_constants.RISK_SCHEMA_V3)
print(json.dumps(schema, indent=2)[:4000])

---
# D · What the model does, and what it says

One call. **Nothing edits the score afterwards.** There were floors, a cap and a
sanctions gate; they are gone. The single assignment to `score` in the whole
backend is the model's own `score_12m`, rescaled.

In [ ]:
llm_output = langchain_llm.country_llm_score(
    country_display=NAME,        # masked to "the country" inside the call
    payload=evidence,            # the three-ledger payload, not the panel one
    articles=scored,
    as_of=AS_OF,
    fulltext_ids=fulltext_ids,
    mask_iso2=ISO2,              # <- production. None would be the named twin.
)

# A failed call returns the no-score shape rather than raising, so nothing
# downstream would notice on its own.
assert llm_output["score"] is not None, "scoring call failed - see the log above"
print(f"{NAME}  {AS_OF}   risk {llm_output['score']:.2f} (12m)  "
      f"{llm_output['score_3m']:.2f} (3m)"
      f"{'   RESTRICTED' if llm_output['non_investable'] else ''}\n")
print(llm_output["bullet_summary"])

### D1 · The raw return, whole

In [ ]:
print(json.dumps(llm_output, indent=2, ensure_ascii=False, default=str))

### D2 · The 0-100 boundary

The prompt asks for **integers 0-100**, because that grid has the rank
resolution the roster needs. Everything downstream — the database, the
front-end — speaks 0-1. `_from_100` runs the moment the call returns, so a
0-100 number never leaves `langchain_llm.py`.

`raw_score_12m` equals `score` by construction now that nothing gates it. The
column is still written so a p1.0 row and a p2.0 row stay comparable.

In [ ]:
print(f"score (12m)      {llm_output['score']}      raw_score_12m {llm_output['raw_score_12m']}")
print(f"score_3m         {llm_output['score_3m']}      raw_score_3m  {llm_output['raw_score_3m']}")
print(f"evidence_coverage {llm_output['evidence_coverage']}\n")

pd.DataFrame([
    {"ledger": k, "0-1": v, "0-100 (as the model said it)": pipeline._to_100(v)}
    for k, v in (llm_output.get("ledger_scores") or {}).items()
]).set_index("ledger")

### D3 · Its reasoning, in its own citations

`subscore_evidence` is the model naming which indicators and which article ids
moved each ledger. `condition_flags` are the binary readings it was asked for.
`news_article_scores` is the per-article impact that drives Top-3 selection.

In [ ]:
print("condition_flags:")
for k, v in (llm_output.get("condition_flags") or {}).items():
    print(f"  {k:<28} {v}")

print("\nsubscore_evidence:")
for ledger, cites in (llm_output.get("subscore_evidence") or {}).items():
    print(f"  {ledger}: {cites}")

if llm_output.get("legal_gate"):
    print(f"\nlegal_gate: {json.dumps(llm_output['legal_gate'], indent=2)}")

pd.DataFrame(llm_output.get("news_article_scores") or []).sort_values(
    "impact", ascending=False).set_index("id")

### D4 · Did it work out which country it was?

The identifiability meter, not a gate. A confident correct guess does not stop
the snapshot — it could not: the US is expected to be identified nearly always
from coverage volume alone, and refusing to score the US would be answering the
wrong question.

Production samples one country in six and records the result. This calls
`probe.probe` directly, which writes nothing.

In [ ]:
RUN_PROBE = True    # one extra cheap-model call

if RUN_PROBE and os.getenv("OPENAI_API_KEY"):
    guess = probe.probe(rewrite.mask_items(scored, ISO2), os.getenv("OPENAI_API_KEY"),
                        fulltext_ids=fulltext_ids)
    hit = guess.get("country") == ISO2
    # ZZ is "no guess". It means either the model said it could not tell, or the
    # probe call itself failed - a failed measurement must not read as a clean one.
    failed = str(guess.get("evidence", "")).startswith("probe failed")
    print(f"guessed {guess.get('country')} at {guess.get('confidence', 0):.2f} confidence"
          f"   (actual {ISO2} - "
          f"{'PROBE FAILED, not a measurement' if failed else 'IDENTIFIED' if hit else 'not identified'})")
    print(f"alternatives: {guess.get('alternatives')}")
    print(f"evidence:     {guess.get('evidence')}")
else:
    print("skipped")

---
# E · After the model

Three things happen to the score before it is stored, and none of them change
it.

### E1 · Lint — contradictions, written down and corrected by nobody

Advisory by design. It records where the model's flags disagree with its
scores; nothing here edits a number, and a lint failure must never cost the
country its snapshot. Findings normally land in `risk_snapshot` itself — this
cell does not write.

In [ ]:
findings = lint.check(
    country_iso2=ISO2,
    as_of=AS_OF,
    condition_flags=llm_output.get("condition_flags"),
    # lint's tripwires are on the model's 0-100 scale; the stored values are 0-1.
    score_3m=pipeline._to_100(llm_output.get("score_3m")),
    score_12m=pipeline._to_100(llm_output.get("score")),
    ledger_scores={k: pipeline._to_100(v)
                   for k, v in (llm_output.get("ledger_scores") or {}).items()},
    suppressed_vol_flag=evidence.get("uncertainty_inputs", {})
                               .get("suppressed_vol_flag", {}).get("value"),
    non_investable=bool(llm_output.get("non_investable")),
)
lint.log_findings(findings)
print(f"{len(findings)} finding(s)" if findings else "no findings - flags and scores agree")

### E2 · Provenance — the hash of what the model actually saw

`items=scored`, not `items`: the manifest's whole promise is that it hashes the
bytes the model saw, and under masking those are not the bytes in the database.
The masking block carries the versions a rebuild needs — without the map's
version the same articles re-mask differently and the row cannot be reproduced.

In [ ]:
manifest = provenance.build_input_manifest(
    items=scored,
    prompt_entries=langchain_llm.prompt_entries(scored),
    fulltext_ids=fulltext_ids,
    payload=panel,
    evidence=evidence,
    payload_health=llm_payload.payload_health(evidence, series, AS_OF, scored),
    model_id=llm_output.get("model_id"),
    prompt_version=llm_output.get("prompt_version"),
    policy_version=llm_output.get("policy_version"),
    seed=ai_client.SEED,
    masking={
        "scoring_mode": "masked",
        "mask_map_version": gazetteer.MASK_MAP_VERSION,
        "gazetteer_version": gazetteer.GAZETTEER_VERSION,
        "sweep_version": rewrite.SWEEP_VERSION,
        # "clean" by construction: assert_clean in C1 raises before sending.
        "mask_integrity_status": "clean",
        "structural_fields": len(evidence.get("structural") or {}),
        "identifiability": None,   # production stores the probe here, 1 country in 6
    },
)
print(json.dumps(manifest, indent=2, default=str)[:3000])

### E3 · Top three — what reaches the dashboard

The model's per-article impact scores plus its topic clustering pick three
articles, one per topic group where it can. These are ranked off the **unmasked**
`items`, because they are shown to a human who knows perfectly well which
country they are reading about.

In [ ]:
imp_map, topic_map = article_ranking.impact_topic_maps(llm_output)
items_by_id = {it["id"]: it for it in items if it.get("id")}
top_ids = article_ranking.select_top_ids(items_by_id, imp_map, topic_map, ISO2)

article_enrichment.enrich_top_images(top_ids, items_by_id)   # scraper, top 3 only
top_articles = article_ranking.build_top_articles(top_ids, items_by_id, imp_map)

pd.DataFrame([
    {"id": a.get("id"), "impact": a.get("impact"), "topic": topic_map.get(a.get("id")),
     "image": bool(a.get("image")), "title": a.get("title")}
    for a in top_articles
]).set_index("id")

## Where this stops

The next line in `pipeline._process_country` is `data_push.upsert_snapshot(...)`,
which writes `risk_snapshot`, `risk_snapshot_article`, `indicator` and
`yearly_value`. This notebook does not call it, so nothing on the dashboard
moved.

In [ ]:
print(f"{NAME} ({ISO2})   as_of {AS_OF}")
print(f"  risk        {llm_output['score']:.2f} (12m)   {llm_output['score_3m']:.2f} (3m)"
      f"{'   RESTRICTED' if llm_output['non_investable'] else ''}")
print(f"  ledgers     " + "  ".join(f"{k}={v:.2f}" for k, v in llm_output["ledger_scores"].items()))
print(f"  evidence    {sum(len(evidence.get(s, {})) for s in LEDGERS)} indicators, "
      f"{len(evidence['computed'])} computed metrics")
print(f"  news        {len(items)} articles, {len(fulltext_ids)} read in full, "
      f"{len(top_articles)} published")
print(f"  stamped     {llm_output['model_id']} / {llm_output['prompt_version']} / "
      f"{llm_output['policy_version']} / seed {ai_client.SEED}")
print(f"  masked      map {gazetteer.MASK_MAP_VERSION}, gate clean")
print(f"\n  written to the database: nothing (digest + rewrite caches only)")